# Machine Learning Models: Interactive Guide & Toy-Dataset Examples

Welcome! This notebook is designed so you can **either read it like a guided textbook** or **run every cell** in **Google Colab**, **Kaggle**, or local **Jupyter**.

Every model uses built-in **Scikit-Learn toy datasets** (`make_regression`, `make_circles`, `make_moons`, `make_blobs`, `load_iris`, `load_breast_cancer`) so there are no external files to download.

### Core Workflow Used Throughout
1. **Split the data (`train_test_split`):** Hold out 20–25% of the data as a test set so we measure how well the model generalizes to *unseen* examples, rather than just memorizing the training set.
2. **Prevent Data Leakage (`make_pipeline`):** Whenever we scale features (`StandardScaler`), we fit the scaler *only* on `X_train` and apply that same transformation to `X_test`.
3. **Evaluate Honestly:** Use RMSE (Root Mean Squared Error) for regression and Accuracy for classification.

> **Setup (if needed):** `!pip install -q scikit-learn xgboost matplotlib`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, root_mean_squared_error

np.random.seed(42)


---
## 1. Linear Regression (Ordinary Least Squares)

### How It Works
Linear Regression predicts a continuous number $\hat{y}$ as a weighted sum of input features:
$$\hat{y} = w_1 x_1 + w_2 x_2 + \dots + b$$
During `.fit()`, **Ordinary Least Squares (OLS)** finds the slope $w$ and intercept $b$ that minimize the **Mean Squared Error (MSE)** — the average squared vertical distance between the real points and the line.

### What This Example Shows
We generate 100 noisy points around a line using `make_regression`, train on 80 points, and plot the fitted red OLS line against the 20 unseen test points.
- **When to use:** Fast, interpretable baseline when relationships are roughly linear.
- **Limitation:** Sensitive to outliers (because errors are squared) and cannot capture curves on its own.


In [ ]:
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

# Toy regression dataset (1 feature, 100 samples)
X, y = make_regression(n_samples=100, n_features=1, noise=15.0, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

y_pred = lin_reg.predict(X_test)
print(f"Linear Regression RMSE: {root_mean_squared_error(y_test, y_pred):.2f}")
print(f"Learned Weight (slope): {lin_reg.coef_[0]:.2f}, Bias (intercept): {lin_reg.intercept_:.2f}")

plt.scatter(X_test, y_test, label="Actual (Test)", alpha=0.8)
plt.plot(X_test, y_pred, color="red", linewidth=2, label="OLS Line")
plt.title("Linear Regression on Toy Data")
plt.xlabel("Feature x"); plt.ylabel("Target y")
plt.legend(); plt.show()


---
## 2. Polynomial Regression (Basis Expansion)

### How It Works
What if the true relationship is curved (like a parabola)? Instead of abandoning linear models, **Polynomial Regression** creates new features $[x, x^2, \dots, x^d]$ using `PolynomialFeatures`, and then feeds them into `LinearRegression`. Because the model is still *linear in the weights* ($w_2 x^2 + w_1 x + b$), it solves cleanly and fast!

### What This Example Shows
We generate data following $y = 2x^2 - 3x + \text{noise}$. A straight line would miss the U-shape completely (high bias / underfitting), while a degree-2 pipeline traces the curve smoothly.
- **Watch out:** Setting `degree` too high (e.g., `degree=15`) makes the curve wiggle wildly to hit training noise (overfitting).


In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Toy curved dataset: y = 2*x^2 - 3*x + noise
X_curve = np.linspace(-3, 3, 120).reshape(-1, 1)
y_curve = 2 * (X_curve[:, 0] ** 2) - 3 * X_curve[:, 0] + np.random.normal(0, 1.5, 120)

X_tr, X_te, y_tr, y_te = train_test_split(X_curve, y_curve, test_size=0.2, random_state=42)

poly_model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    LinearRegression()
)
poly_model.fit(X_tr, y_tr)
y_pred = poly_model.predict(X_te)
print(f"Polynomial (degree=2) RMSE: {root_mean_squared_error(y_te, y_pred):.2f}")

order = np.argsort(X_te[:, 0])
plt.scatter(X_te, y_te, label="Actual (Test)")
plt.plot(X_te[order], y_pred[order], color="orange", linewidth=2, label="Degree-2 Fit")
plt.title("Polynomial Regression on Curved Toy Data")
plt.xlabel("Feature x"); plt.ylabel("Target y")
plt.legend(); plt.show()


---
## 3. Regularization: Ridge ($L_2$) vs. Lasso ($L_1$)

### How It Works
When a dataset has many features (especially noisy or correlated ones), OLS can assign huge, unstable weights. **Regularization** adds a penalty for large weights controlled by `alpha`:
- **Ridge ($L_2$):** Penalizes $\alpha \sum w_j^2$ (a circular constraint). It shrinks all weights smoothly toward zero, keeping every feature in the model.
- **Lasso ($L_1$):** Penalizes $\alpha \sum |w_j|$ (a diamond-shaped constraint). Because the diamond's corners sit on the coordinate axes, Lasso snaps useless weights to **exact zero** — performing **automatic feature selection**.

### What This Example Shows
We create a dataset with **10 features**, where only **3 carry real signal** and 7 are pure noise.
- Ridge keeps all 10 weights non-zero (shrinking the noisy ones).
- Lasso zeroes out the noisy features, leaving ~3 non-zero weights!
- **Important:** Always use `StandardScaler()` before Ridge/Lasso so features on larger scales aren't penalized differently.


In [ ]:
from sklearn.linear_model import Ridge, Lasso

X_reg, y_reg = make_regression(
    n_samples=150, n_features=10, n_informative=3, noise=10.0, random_state=42
)
X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(X_tr, y_tr)
lasso = make_pipeline(StandardScaler(), Lasso(alpha=2.0)).fit(X_tr, y_tr)

print(f"Ridge Test RMSE: {root_mean_squared_error(y_te, ridge.predict(X_te)):.2f}")
print(f"Lasso Test RMSE: {root_mean_squared_error(y_te, lasso.predict(X_te)):.2f}")
print("Ridge non-zero weights:", np.sum(ridge[-1].coef_ != 0), "/ 10")
print("Lasso non-zero weights:", np.sum(lasso[-1].coef_ != 0), "/ 10 (automatic feature selection!)")


---
## 4. Logistic Regression & k-Nearest Neighbors (k-NN)

### How They Work
Both models solve **classification** problems (predicting discrete categories), but in completely different ways:
- **Logistic Regression (Parametric):** Computes a linear score $z = w^T x + b$ and passes it through the **Sigmoid function** $\sigma(z) = \frac{1}{1 + e^{-z}}$, outputting a calibrated probability between `0` and `1`.
- **k-Nearest Neighbors (Non-parametric / Lazy Learner):** Doesn't learn a formula at all! It stores the training points and classifies each new point by taking a **majority vote of its $k$ closest neighbors**.
  - Small $k$ (like `k=1`) creates jagged boundaries sensitive to noise.
  - Larger $k$ (like `k=5`) smooths out the boundary.

### What This Example Shows
We test both on Scikit-Learn's built-in **Breast Cancer** dataset (30 numerical cell features, binary target: malignant vs. benign). Notice how `k=5` typically outperforms `k=1` on unseen test data.


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

# Logistic Regression
log_clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
log_clf.fit(X_tr, y_tr)
print(f"Logistic Regression Accuracy: {accuracy_score(y_te, log_clf.predict(X_te)):.3f}")

# k-NN (comparing k=1 vs k=5)
for k in [1, 5]:
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    knn.fit(X_tr, y_tr)
    print(f"k-NN (k={k}) Accuracy:         {accuracy_score(y_te, knn.predict(X_te)):.3f}")


---
## 5. Naive Bayes Classifier

### How It Works
Naive Bayes applies **Bayes' Theorem** to compute the probability of each class given the observed features:
$$P(y \mid X) \propto P(y) \prod_{i=1}^{m} P(x_i \mid y)$$
It is called **"naive"** because it assumes all features $x_1, \dots, x_m$ are conditionally independent given the class $y$. Even when that assumption isn't strictly true, Naive Bayes is blazing fast and works remarkably well on high-dimensional data (like text spam filtering with `MultinomialNB`).

### What This Example Shows
We use `GaussianNB` (which assumes continuous features follow a bell curve within each class) on the classic 3-class **Iris** flower dataset, and inspect the predicted class probabilities for the first two test flowers.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.naive_bayes import GaussianNB

iris = load_iris()
X_tr, X_te, y_tr, y_te = train_test_split(iris.data, iris.target, test_size=0.25, random_state=42)

nb = GaussianNB().fit(X_tr, y_tr)
print(f"Gaussian Naive Bayes Accuracy (Iris): {accuracy_score(y_te, nb.predict(X_te)):.3f}")
print("Sample predicted probabilities (3 classes):\n", np.round(nb.predict_proba(X_te[:2]), 3))


---
## 6. Support Vector Machines (SVM): Linear vs. RBF Kernel Trick

### How It Works
An **SVM** searches for the **maximum-margin hyperplane** — the widest possible "street" separating two classes. Only the closest edge points (**Support Vectors**) determine where this street sits.
- **Linear SVM (`kernel="linear"`):** Draws a straight line/plane. Fails if the classes are tangled in a circle or spiral.
- **The Kernel Trick (`kernel="rbf"`):** Implicitly measures similarity in a higher-dimensional space where concentric rings *can* be sliced apart by a flat plane — without ever explicitly computing the higher-dimensional coordinates!

### What This Example Shows
On `make_circles` (one ring inside another), a Linear SVM gets ~50% accuracy (no better than a coin flip), while the **RBF Kernel SVM** jumps to ~95–100%!


In [ ]:
from sklearn.datasets import make_circles
from sklearn.svm import SVC

X_circ, y_circ = make_circles(n_samples=300, factor=0.4, noise=0.08, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_circ, y_circ, test_size=0.25, random_state=42)

svm_lin = make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0)).fit(X_tr, y_tr)
svm_rbf = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale")).fit(X_tr, y_tr)

print(f"Linear SVM Accuracy (Circles): {accuracy_score(y_te, svm_lin.predict(X_te)):.3f} (cannot separate rings!)")
print(f"RBF Kernel SVM Accuracy:       {accuracy_score(y_te, svm_rbf.predict(X_te)):.3f} (kernel trick works!)")


---
## 7. Decision Trees (CART) & Pruning

### How It Works
A **Decision Tree** works like a flowchart of yes/no threshold questions (e.g., `mean radius <= 14.2?`). At each step, the CART algorithm picks the split that maximizes **node purity** (minimizing Gini Impurity or Entropy).
- **Strength:** No feature scaling required, handles non-linearities, and is completely human-readable (`export_text`).
- **Overfitting Risk:** An unconstrained tree keeps splitting until every leaf has 1 sample — achieving **100% training accuracy** by memorizing noise, which hurts test accuracy. **Pruning** (`max_depth=3`, `min_samples_leaf=5`) stops growth early so the rules generalize.

### What This Example Shows
Compare the Train vs. Test gap between `tree_full` (unpruned) and `tree_pruned` (`max_depth=3`), and read the actual learned decision rules!


In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

tree_full = DecisionTreeClassifier(random_state=42).fit(X_tr, y_tr)
tree_pruned = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=42).fit(X_tr, y_tr)

print(f"Unpruned Tree -> Train: {tree_full.score(X_tr, y_tr):.3f} | Test: {tree_full.score(X_te, y_te):.3f}")
print(f"Pruned Tree   -> Train: {tree_pruned.score(X_tr, y_tr):.3f} | Test: {tree_pruned.score(X_te, y_te):.3f}")
print("\nTop splits of pruned tree:")
print(export_text(tree_pruned, feature_names=list(data.feature_names), max_depth=2))


---
## 8. Ensemble Models: Random Forests & Gradient Boosted Trees (GBDT / XGBoost)

### How They Work
Single decision trees have high variance. **Ensembles** combine hundreds of trees in two complementary ways:
1. **Random Forest (Bagging — Parallel):** Trains many independent trees on random bootstrap samples of rows *and* random subsets of features (`max_features="sqrt"`), then averages their votes. The unused ~37% of rows for each tree provide a free **Out-of-Bag (OOB)** validation score.
2. **Gradient Boosting / XGBoost (Boosting — Sequential):** Trains shallow trees *one after another*, where each new tree specifically predicts and corrects the **residual errors** left by the previous trees:
   $$F_m(x) = F_{m-1}(x) + \text{learning\_rate} \cdot h_m(x)$$

### What This Example Shows
We compare Random Forest, Scikit-Learn's `GradientBoostingClassifier`, and `XGBClassifier` on the Breast Cancer dataset.


In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

rf = RandomForestClassifier(n_estimators=150, max_features="sqrt", oob_score=True, random_state=42)
rf.fit(X_tr, y_tr)
print(f"Random Forest -> OOB: {rf.oob_score_:.3f} | Test: {rf.score(X_te, y_te):.3f}")

gb = GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42)
gb.fit(X_tr, y_tr)
print(f"Sklearn GBDT  -> Test: {gb.score(X_te, y_te):.3f}")

try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, eval_metric="logloss", random_state=42)
    xgb.fit(X_tr, y_tr)
    print(f"XGBoost       -> Test: {xgb.score(X_te, y_te):.3f}")
except ImportError:
    print("XGBoost not installed (run: pip install xgboost)")


---
## 9. Unsupervised Clustering: K-Means vs. DBSCAN

In **unsupervised learning**, there are no `y` labels — the algorithm must discover structure on its own.
- **K-Means:** You pick $K$ in advance. It drops $K$ centroids, assigns every point to its nearest centroid, moves each centroid to the cluster mean, and repeats until convergence. Great for **round, blob-like clusters**.
- **DBSCAN:** You pick a neighborhood radius `eps` and `min_samples`. It chains together dense regions, so it can trace **arbitrary curved shapes** (like two interlocking moons) and automatically flags isolated outliers as **Noise (`-1`)**.

### What This Example Shows
Side-by-side plots of K-Means separating 3 spherical blobs (`make_blobs`) and DBSCAN cleanly tracing 2 interlocking moons (`make_moons`).


In [ ]:
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN

# 1. K-Means on spherical blobs
X_blobs, _ = make_blobs(n_samples=250, centers=3, cluster_std=0.7, random_state=42)
km_labels = KMeans(n_clusters=3, n_init="auto", random_state=42).fit_predict(X_blobs)

# 2. DBSCAN on non-spherical moons
X_moons, _ = make_moons(n_samples=250, noise=0.06, random_state=42)
X_moons_scaled = StandardScaler().fit_transform(X_moons)
db_labels = DBSCAN(eps=0.3, min_samples=5).fit_predict(X_moons_scaled)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=km_labels, cmap="viridis")
axes[0].set_title("K-Means (3 Spherical Blobs)")
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=db_labels, cmap="plasma")
axes[1].set_title(f"DBSCAN (Moons, noise points={np.sum(db_labels == -1)})")
plt.tight_layout(); plt.show()


---
## 10. Principal Component Analysis (PCA)

### How It Works
**PCA** is a dimensionality reduction technique. When you have many correlated features, PCA rotates the coordinate axes to find new perpendicular directions (**Principal Components**):
- **PC1** points in the direction of **maximum variance** (the widest spread of the data).
- **PC2** is orthogonal to PC1 and captures the next largest amount of variance.

By keeping only the top $k$ components, we compress the dataset while losing as little information as possible. Always apply `StandardScaler` first so features with larger units don't dominate PC1.

### What This Example Shows
We compress the **4D Iris dataset** (sepal length/width, petal length/width) down to **2D** and print `explained_variance_ratio_` — showing that just 2 components retain **~95.8%** of the original information!


In [ ]:
from sklearn.decomposition import PCA

pca_pipe = make_pipeline(StandardScaler(), PCA(n_components=2, random_state=42))
X_iris_2d = pca_pipe.fit_transform(iris.data)

var_ratio = pca_pipe[-1].explained_variance_ratio_
print(f"Explained Variance: PC1={var_ratio[0]:.2%}, PC2={var_ratio[1]:.2%}, Total Retained={var_ratio.sum():.2%}")

plt.scatter(X_iris_2d[:, 0], X_iris_2d[:, 1], c=iris.target, cmap="viridis", edgecolor="k")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Iris Dataset Compressed from 4D to 2D via PCA")
plt.show()
